# DR Preprocessing — APTOS/Kaggle 2015 Dataset

Builds a clean, reproducible train/val split from the **raw** 2015 diabetic retinopathy dataset
(no dependency on any previously-generated Kaggle output).

**Pipeline:**
1. Download the raw dataset from Kaggle
2. Crop + resize + sharpen every image (straight to 528×528, matching the EfficientNetB6 input size)
3. One stratified 85/15 train/val split, fixed random seed (reproducible, no leakage)
4. Offline data augmentation on the **train split only**, to balance every minority class up to
   the majority class's count
5. Copy the final `train/` and `val/` folders to Google Drive so they persist across Colab sessions

> Note: this preprocessing function is duplicated in `02_preprocessing_2019.ipynb` for now since
> each notebook needs to stand alone in Colab. When this moves to GitHub, pull both copies into a
> single `src/preprocessing.py` module that both notebooks import.


## 0. Download the dataset from Kaggle

In [9]:
import json
import os
from google.colab import userdata

os.makedirs('/root/.kaggle', exist_ok=True)

kaggle_json = {
    "username": userdata.get("KAGGLE_USERNAME"),
    "key": userdata.get("KAGGLE_KEY"),
}
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_json, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

!pip install -q kaggle


In [10]:
!kaggle datasets download -d sovitrath/diabetic-retinopathy-2015-data-colored-resized \
    -p /content/data_2015 --unzip


Dataset URL: https://www.kaggle.com/datasets/sovitrath/diabetic-retinopathy-2015-data-colored-resized
License(s): CC0-1.0
100% 1.94G/1.94G [00:24<00:00, 83.7MB/s]



In [11]:
# Sanity check: confirm the extracted folder structure before hardcoding a path below.
# Kaggle zip layouts occasionally nest an extra folder level -- adjust DATA_ROOT if this doesn't
# show the 5 class folders directly.
for root, dirs, files_ in os.walk('/content/data_2015'):
    depth = root.count(os.sep) - '/content/data_2015'.count(os.sep)
    if depth <= 2:
        print(root, '->', dirs[:10])


/content/data_2015 -> ['colored_images']
/content/data_2015/colored_images -> ['colored_images']
/content/data_2015/colored_images/colored_images -> ['Mild', 'Severe', 'Moderate', 'No_DR', 'Proliferate_DR']


## 1. Imports and preprocessing functions

In [12]:
import cv2
import numpy as np
from tqdm import tqdm
from glob import glob
from sklearn.model_selection import train_test_split
import albumentations as A
import shutil

IMG_SIZE = 528  # matches EfficientNetB6 input size

folder_to_class = {
    "No_DR": 0,
    "Mild": 1,
    "Moderate": 2,
    "Severe": 3,
    "Proliferate_DR": 4,
}

# Update this if the sanity-check cell above showed a different structure
DATA_ROOT = "/content/data_2015/colored_images/colored_images"


In [13]:
def crop_image_from_gray(img, tol=7):
    """Crop the near-black border around the fundus photo."""
    if img.ndim == 2:
        mask = img > tol
        return img[np.ix_(mask.any(1), mask.any(0))]
    elif img.ndim == 3:
        gray_img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        mask = gray_img > tol
        if mask.any():
            img1 = img[:, :, 0][np.ix_(mask.any(1), mask.any(0))]
            img2 = img[:, :, 1][np.ix_(mask.any(1), mask.any(0))]
            img3 = img[:, :, 2][np.ix_(mask.any(1), mask.any(0))]
            img = np.stack([img1, img2, img3], axis=-1)
        return img
    return img


def preprocess_image(image_path, desired_size=IMG_SIZE):
    """Crop black border, resize, and apply the Ben Graham-style unsharp mask."""
    im = cv2.imread(image_path)
    if im is None:
        return None
    im = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
    im = crop_image_from_gray(im)
    if im is None or im.shape[0] == 0 or im.shape[1] == 0:
        return None
    im = cv2.resize(im, (desired_size, desired_size))
    res = cv2.addWeighted(im, 4.5, cv2.GaussianBlur(im, (0, 0), 10), -4, 100)
    return res


## 2. Build the file list from the raw dataset and split train/val

A single stratified 85/15 split, done once, from the raw source images only -- this is what eliminates the leakage risk in the old notebooks (where the "test set" was quietly topped up with images from elsewhere).

In [14]:
samples = []  # (path, label)
for folder, label in folder_to_class.items():
    class_dir = os.path.join(DATA_ROOT, folder)
    for fname in os.listdir(class_dir):
        samples.append((os.path.join(class_dir, fname), label))

print(f"Total raw images found: {len(samples)}")

paths = [s[0] for s in samples]
labels = [s[1] for s in samples]

train_paths, val_paths, train_labels, val_labels = train_test_split(
    paths, labels, test_size=0.15, stratify=labels, random_state=42
)

print(f"Train: {len(train_paths)} | Val: {len(val_paths)}")


Total raw images found: 35126
Train: 29857 | Val: 5269


## 3. Preprocess and save both splits

In [15]:
TRAIN_DIR = "/content/preprocessed_2015/train"
VAL_DIR = "/content/preprocessed_2015/val"


def process_and_save(path_label_list, out_dir):
    for cls in folder_to_class.values():
        os.makedirs(os.path.join(out_dir, str(cls)), exist_ok=True)

    for path, label in tqdm(path_label_list):
        img = preprocess_image(path)
        if img is None:
            continue
        fname = os.path.basename(path)
        save_path = os.path.join(out_dir, str(label), fname)
        cv2.imwrite(save_path, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))


process_and_save(list(zip(train_paths, train_labels)), TRAIN_DIR)
process_and_save(list(zip(val_paths, val_labels)), VAL_DIR)


100%|██████████| 5269/5269 [03:51<00:00, 22.77it/s]


In [16]:
def count_classes(directory):
    counts = {}
    for cls in sorted(os.listdir(directory), key=int):
        class_dir = os.path.join(directory, cls)
        counts[int(cls)] = len(os.listdir(class_dir))
    return counts


print("Train class counts (before augmentation):", count_classes(TRAIN_DIR))
print("Val class counts:  ", count_classes(VAL_DIR))


Train class counts (before augmentation): {0: 21938, 1: 2077, 2: 4498, 3: 742, 4: 602}
Val class counts:   {0: 3872, 1: 366, 2: 794, 3: 131, 4: 106}


## 4. Offline augmentation -- balance minority classes (train split only)

Every class below the majority class's count gets extra *augmented* copies (flip / rotate / brightness-contrast / hue-saturation) generated from its own existing images, until it matches the majority class. This never touches `val/`, so validation numbers stay honest.

Note: the training notebook also applies its own random augmentation (`train_aug`) live during training. That's fine -- it'll just augment these already-balanced images further and differently each epoch.

In [17]:
balance_aug = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.4),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=20, p=0.7),
    A.HueSaturationValue(p=0.3),
])

VALID_EXTS = ('.jpg', '.jpeg', '.png')


def augment_and_save(class_dir, num_to_add):
    existing_files = [f for f in os.listdir(class_dir) if f.lower().endswith(VALID_EXTS)]
    for i in tqdm(range(num_to_add)):
        src_name = existing_files[i % len(existing_files)]
        img = cv2.imread(os.path.join(class_dir, src_name))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        augmented = balance_aug(image=img_rgb)['image']
        save_name = f"aug_{i}_{src_name}"
        cv2.imwrite(os.path.join(class_dir, save_name), cv2.cvtColor(augmented, cv2.COLOR_RGB2BGR))


/usr/local/lib/python3.13/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [18]:
train_counts = count_classes(TRAIN_DIR)
majority_count = max(train_counts.values())

for cls, count in train_counts.items():
    deficit = majority_count - count
    if deficit > 0:
        print(f"Class {cls}: {count} -> augmenting {deficit} extra images to reach {majority_count}")
        augment_and_save(os.path.join(TRAIN_DIR, str(cls)), deficit)

print("\nTrain class counts (after augmentation):", count_classes(TRAIN_DIR))


Class 1: 2077 -> augmenting 19861 extra images to reach 21938


100%|██████████| 19861/19861 [09:02<00:00, 36.63it/s]


Class 2: 4498 -> augmenting 17440 extra images to reach 21938


100%|██████████| 17440/17440 [08:01<00:00, 36.19it/s]


Class 3: 742 -> augmenting 21196 extra images to reach 21938


100%|██████████| 21196/21196 [09:48<00:00, 36.03it/s]


Class 4: 602 -> augmenting 21336 extra images to reach 21938


100%|██████████| 21336/21336 [09:45<00:00, 36.42it/s]



Train class counts (after augmentation): {0: 21938, 1: 21938, 2: 21938, 3: 21938, 4: 21938}


## 5. Persist to Google Drive

In [22]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/MyDrive/DR_Screening/2015"
os.makedirs(DRIVE_ROOT, exist_ok=True)

# Zip locally first (fast, real disk) instead of copying thousands of files
# one-by-one over the Drive FUSE mount (slow).
shutil.make_archive("/content/preprocessed_2015_train", 'zip', TRAIN_DIR)
shutil.make_archive("/content/preprocessed_2015_val", 'zip', VAL_DIR)

# Now just move two files to Drive -- this is the part that's actually fast.
shutil.copy("/content/preprocessed_2015_train.zip", DRIVE_ROOT)
shutil.copy("/content/preprocessed_2015_val.zip", DRIVE_ROOT)

print("Saved to:", DRIVE_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


KeyboardInterrupt: 

## Summary

- Raw images pulled fresh from Kaggle, no dependency on any earlier notebook's output.
- One stratified 85/15 split with a fixed seed -- reproducible, no leakage between train and val.
- Preprocessed straight to 528×528 (crop border + resize + unsharp mask), matching EfficientNetB6's input size.
- Train split rebalanced via offline augmentation so every class matches the majority class count; val left untouched.
- Final data lives at `/content/drive/MyDrive/DR_data/2015/{train,val}`.

**Still open:** the training notebook (`DR_Screening_final.ipynb`) currently points at
`preprocessed_train2015/preprocessed_train15` and `preprocessed_test2015/preprocessed_test` --
when we rebuild the training notebook, its paths need to be updated to match `2015/train` and `2015/val` above.
